In [6]:
from itertools import product



def Not(a):        return ('not', a)
def And(a, b):     return ('and', a, b)
def Or(a, b):      return ('or', a, b)
def Implies(a, b): return ('implies', a, b)
def Iff(a, b):     return ('iff', a, b)


def to_string(s):
    """Convert a sentence to readable text, e.g. (Q -> P)."""
    if isinstance(s, str):
        return s
    if s[0] == 'not':
        return '~' + to_string(s[1])
    symbol = {'and': '&', 'or': '|', 'implies': '->', 'iff': '<->'}[s[0]]
    return '(' + to_string(s[1]) + ' ' + symbol + ' ' + to_string(s[2]) + ')'


def get_symbols(s):
    """Return the set of proposition symbols used in a sentence."""
    if isinstance(s, str):
        return {s}
    symbols = set()
    for part in s[1:]:
        symbols |= get_symbols(part)
    return symbols



def pl_true(s, model):
    if isinstance(s, str):                      # symbol -> look up its value
        return model[s]
    op = s[0]
    if op == 'not':
        return not pl_true(s[1], model)
    if op == 'and':
        return pl_true(s[1], model) and pl_true(s[2], model)
    if op == 'or':
        return pl_true(s[1], model) or pl_true(s[2], model)
    if op == 'implies':                         # a -> b  is  ~a | b
        return (not pl_true(s[1], model)) or pl_true(s[2], model)
    if op == 'iff':                             # same value on both sides
        return pl_true(s[1], model) == pl_true(s[2], model)


def kb_true(kb, model):
    """KB is the AND of all its sentences."""
    return all(pl_true(sentence, model) for sentence in kb)

def tt_entails(kb, alpha):
    symbols = set(get_symbols(alpha))
    for sentence in kb:
        symbols |= get_symbols(sentence)
    return tt_check_all(kb, alpha, sorted(symbols), {})


def tt_check_all(kb, alpha, symbols, model):
    if not symbols:                             # all symbols assigned = one row
        if kb_true(kb, model):
            return pl_true(alpha, model)        # KB true -> alpha must be true
        return True                             # KB false -> row does not matter
    p, rest = symbols[0], symbols[1:]
    return (tt_check_all(kb, alpha, rest, {**model, p: True}) and
            tt_check_all(kb, alpha, rest, {**model, p: False}))

def print_truth_table(kb, alpha):
    symbols = set(get_symbols(alpha))
    for sentence in kb:
        symbols |= get_symbols(sentence)
    symbols = sorted(symbols)

    headers = symbols + [to_string(s) for s in kb] + ['KB', to_string(alpha)]
    widths = [max(len(h), 5) for h in headers]
    line = ' | '.join(h.center(w) for h, w in zip(headers, widths))
    print(line)
    print('-' * len(line))

    kb_models = []
    for values in product([False, True], repeat=len(symbols)):
        model = dict(zip(symbols, values))
        row = [model[s] for s in symbols]
        row += [pl_true(s, model) for s in kb]
        row += [kb_true(kb, model), pl_true(alpha, model)]
        text = ' | '.join(('T' if v else 'F').center(w) for v, w in zip(row, widths))
        if kb_true(kb, model):
            text += '   <-- KB is true (model of KB)'
            kb_models.append(model)
        print(text)

    print('\nModels in which KB is true:')
    for m in kb_models:
        print('   ', m, '-> query is', pl_true(alpha, m))


def check(kb, alpha):
    print('\n' + '=' * 70)
    print('KB    :', ',  '.join(to_string(s) for s in kb))
    print('Query :', to_string(alpha))
    print('=' * 70)
    print_truth_table(kb, alpha)
    result = tt_entails(kb, alpha)
    print('\nDoes KB entail', to_string(alpha), '?  ->', result)

if __name__ == '__main__':
    KB = [Implies('Q', 'P'),
          Implies('P', Not('Q')),
          Or('Q', 'R')]

    check(KB, 'R')                      # ii)  KB |= R ?
    check(KB, Implies('R', 'P'))        # iii) KB |= R -> P ?
    check(KB, Implies('Q', 'R'))        # iv)  KB |= Q -> R ?


KB    : (Q -> P),  (P -> ~Q),  (Q | R)
Query : R
  P   |   Q   |   R   | (Q -> P) | (P -> ~Q) | (Q | R) |   KB  |   R  
----------------------------------------------------------------------
  F   |   F   |   F   |    T     |     T     |    F    |   F   |   F  
  F   |   F   |   T   |    T     |     T     |    T    |   T   |   T     <-- KB is true (model of KB)
  F   |   T   |   F   |    F     |     T     |    T    |   F   |   F  
  F   |   T   |   T   |    F     |     T     |    T    |   F   |   T  
  T   |   F   |   F   |    T     |     T     |    F    |   F   |   F  
  T   |   F   |   T   |    T     |     T     |    T    |   T   |   T     <-- KB is true (model of KB)
  T   |   T   |   F   |    T     |     F     |    T    |   F   |   F  
  T   |   T   |   T   |    T     |     F     |    T    |   F   |   T  

Models in which KB is true:
    {'P': False, 'Q': False, 'R': True} -> query is True
    {'P': True, 'Q': False, 'R': True} -> query is True

Does KB entail R ?  -> True

KB    